# 01 Bronze - Ingest Historical Ads

This notebook mirrors the local Python ingestion layer.

Local equivalent:

- `src/skill_observatory/ingestion/pipelines/load_historical_ads.py`
- DuckDB table: `historical_job_ads`

Fabric role:

- read uploaded Parquet files from OneLake
- normalize column types for downstream SQL endpoint compatibility
- write a Bronze Delta table in the Lakehouse

In [ ]:
from pyspark.sql import functions as F

BRONZE_SOURCE_PATH = "Files/bronze/historical_job_ads"
BRONZE_TABLE = "historical_job_ads_bronze"

print(f"Reading uploaded Bronze files from {BRONZE_SOURCE_PATH}")
bronze_df = spark.read.parquet(BRONZE_SOURCE_PATH)

print("Source schema:")
bronze_df.printSchema()

In [ ]:
# Keep the Bronze table close to the local DuckDB contract, but cast dates
# to Fabric-friendly types before the SQL endpoint reads the table.
bronze_clean = (
    bronze_df
    .withColumn("publication_date", F.to_timestamp("publication_date"))
    .withColumn("publication_month", F.to_date("publication_month"))
)

bronze_clean.write.mode("overwrite").format("delta").saveAsTable(BRONZE_TABLE)

print(f"Wrote Bronze table: {BRONZE_TABLE}")
print(f"Rows: {bronze_clean.count():,}")

In [ ]:
display(
    spark.table(BRONZE_TABLE)
    .select("id", "publication_month", "headline", "municipality", "region")
    .limit(20)
)